In [ ]:
import subprocess, sys

def _require(package, import_name=None):
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

_require("pandas")
_require("numpy")
_require("matplotlib")
_require("earthengine-api", "ee")
_require("geemap")
_require("scikit-learn", "sklearn")
_require("rasterio")

import os, math, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import ee
import geemap

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer

import rasterio

#######################################################################
# GLOBAL CONSTANTS
#######################################################################

# Saskatchewan prairies — 2° × 2° box centred near Saskatoon
# Wide enough to capture all target crop classes across the landscape
TRAIN_AOI_COORDS = [-107.5, 51.0, -105.5, 53.0]
PRED_AOI_COORDS  = [-107.5, 51.0, -105.5, 53.0]

TRAIN_YEAR        = 2021       # year used for training data
PRED_YEAR         = 2022       # year used for prediction / validation
NUM_SAMPLES_TRAIN = 100000
NUM_SAMPLES_PRED  = 200000
SAMPLE_SCALE      = 30        # metres — matches AAFC/ACI native resolution

# AAFC Annual Crop Inventory classes — Saskatchewan prairies
# 131=Spring Wheat, 133=Lentils, 134=Dry Peas, 135=Canola, 137=Barley, 500=Other
SPECIFIC_CROPS = [131.0, 133.0, 134.0, 135.0, 137.0, 500.0]
CROP_LABELS    = {131.0: 'Spr Wheat', 133.0: 'Lentils',  134.0: 'Dry Peas',
                  135.0: 'Canola',    137.0: 'Barley',   500.0: 'Other'}
NUM_CLASSES    = len(SPECIFIC_CROPS)

# None = auto-balance to minority class; or set an integer cap per class
SAMPLES_PER_CLASS = None

# LABEL_COL is set dynamically in the GEE setup cell below
# NOTE: AAFC/ACI has no confidence band, so LABEL_COL = NUM_EMBEDDING_BANDS (not +1)

In [ ]:
#######################################################################
# GEE INITIALISATION & SETUP
#######################################################################

ee.Initialize(project='ee-rmccormick314-gfsad')

TRAIN_AOI = ee.Geometry.Rectangle(TRAIN_AOI_COORDS)
PRED_AOI  = ee.Geometry.Rectangle(PRED_AOI_COORDS)

# ── Training Year Data ────────────────────────────────────────────────
train_embedding = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f'{TRAIN_YEAR}-01-01', f'{TRAIN_YEAR}-12-31')
    .filterBounds(TRAIN_AOI)
    .first()
)
train_aafc = (
    ee.ImageCollection("AAFC/ACI")
    .filterDate(f'{TRAIN_YEAR}-01-01', f'{TRAIN_YEAR}-12-31')
    .first()
)

# ── Prediction Year Data ──────────────────────────────────────────────
pred_embedding = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f'{PRED_YEAR}-01-01', f'{PRED_YEAR}-12-31')
    .filterBounds(PRED_AOI)
    .first()
)
pred_aafc = (
    ee.ImageCollection("AAFC/ACI")
    .filterDate(f'{PRED_YEAR}-01-01', f'{PRED_YEAR}-12-31')
    .first()
)

# ── Derive LABEL_COL from embedding dimensionality ────────────────────
EMBEDDING_BANDS     = train_embedding.bandNames().getInfo()
NUM_EMBEDDING_BANDS = len(EMBEDDING_BANDS)
LABEL_COL           = NUM_EMBEDDING_BANDS  # AAFC has no confidence band

print(f"Train year: {TRAIN_YEAR}  |  Pred year: {PRED_YEAR}")
print(f"Embedding bands ({NUM_EMBEDDING_BANDS}): {EMBEDDING_BANDS}")
print(f"LABEL_COL: {LABEL_COL}")

In [ ]:
#######################################################################
# SAMPLE TRAINING DATA FROM GEE  (TRAIN_YEAR)
# NOTE: AAFC/ACI has no confidence band — only embedding + label stacked
#######################################################################

training_image = (
    train_embedding
    .addBands(train_aafc.select('landcover').rename('aafc_label'))
)

print(f"Sampling {NUM_SAMPLES_TRAIN} training pixels (year={TRAIN_YEAR}, scale={SAMPLE_SCALE}m) ...")
training_fc = training_image.sample(
    region    = TRAIN_AOI,
    scale     = SAMPLE_SCALE,
    numPixels = NUM_SAMPLES_TRAIN,
    seed      = 42,
    geometries= False,
    dropNulls = True,
    tileScale = 8       # tiles the computation to stay under GEE memory limits
)

AAFC_df = geemap.ee_to_df(training_fc, remove_geom=False)
AAFC_df = AAFC_df.drop(columns=[c for c in ['geo'] if c in AAFC_df.columns])

col_map = {b: i for i, b in enumerate(EMBEDDING_BANDS)}
col_map['aafc_label'] = LABEL_COL
AAFC_df = AAFC_df.rename(columns=col_map)

print(f"Recovered {len(AAFC_df)} training samples.")
print(AAFC_df.head())

In [ ]:
#######################################################################
# PROCESS TRAINING DATA
#######################################################################

aafc_df = AAFC_df.copy()

# Remap minor classes into target classes
# Winter Wheat (132) → Spring Wheat (131)
aafc_df[LABEL_COL] = np.where(aafc_df[LABEL_COL] == 132.0, 131.0, aafc_df[LABEL_COL])
# Oats (138) + Mixed Grains (139) → Barley (137)
aafc_df[LABEL_COL] = np.where(aafc_df[LABEL_COL] == 138.0, 137.0, aafc_df[LABEL_COL])
aafc_df[LABEL_COL] = np.where(aafc_df[LABEL_COL] == 139.0, 137.0, aafc_df[LABEL_COL])
# Everything else → Other (500)
aafc_df[LABEL_COL] = np.where(
    ~aafc_df[LABEL_COL].isin(SPECIFIC_CROPS), 500.0, aafc_df[LABEL_COL]
)

aafc_df = aafc_df.reset_index(drop=True)

# Show raw counts before balancing so we can spot missing classes
counts = aafc_df[LABEL_COL].value_counts()
print("Raw class counts after remapping:")
print(counts.rename(CROP_LABELS))

# Warn about classes with very few samples
MIN_SAMPLES = 50
sparse = counts[counts < MIN_SAMPLES]
if len(sparse) > 0:
    print(f"\nWARNING: {len(sparse)} class(es) have fewer than {MIN_SAMPLES} samples.")
    print("Consider expanding TRAIN_AOI_COORDS or adding these codes to remapping rules:")
    print(sparse.rename(CROP_LABELS))

# Balance classes — floor at MIN_SAMPLES so balancing doesn't collapse the dataset
n_per_class = SAMPLES_PER_CLASS if SAMPLES_PER_CLASS is not None \
              else max(MIN_SAMPLES, int(counts.min()))

# Only sample up to what each class actually has
aafc_df = pd.concat([
    grp.sample(min(len(grp), n_per_class), random_state=42)
    for _, grp in aafc_df.groupby(LABEL_COL)
]).reset_index(drop=True)

print(f"\nBalanced to {n_per_class} samples per class (floor={MIN_SAMPLES}).")
print(aafc_df[LABEL_COL].value_counts().rename(CROP_LABELS))

aafc_df_all = aafc_df

# Feature / label split
FEATURES = aafc_df.loc[:, aafc_df.columns != LABEL_COL]

from sklearn.experimental import enable_iterative_imputer
imp = SimpleImputer(missing_values=np.nan, strategy='mean')
imp.fit(FEATURES)
FEATURES = imp.transform(FEATURES)

LABELS = aafc_df[LABEL_COL]

# Adaptive test size: at least 3 samples per observed class, never less than NUM_CLASSES
n_classes_found = int(aafc_df[LABEL_COL].nunique())
n_test = max(n_classes_found * 3, int(len(aafc_df) * 0.05))

X_train, X_test, y_train, y_test = train_test_split(
    FEATURES, LABELS,
    test_size    = n_test,
    random_state = 42,
    stratify     = LABELS
)

print(f"\nTrain: {len(X_train)}  Test: {len(X_test)}  Classes found: {n_classes_found}")
print("\nDISTRIBUTION:")
print(pd.DataFrame(y_train).value_counts(normalize=True) * 100)

In [ ]:
aafc_df[LABEL_COL].value_counts()

In [ ]:
#######################################################################
# ESTABLISH AND FIT DATA SCALER
#######################################################################

SCALER_NAME = 'Canada-SK-Scaler'

try:
    scaler = pickle.load(open(SCALER_NAME + ".pickle", "rb"))
    scaler.fit(X_train)
except (OSError, IOError):
    scaler = StandardScaler()
    scaler.fit(X_train)
    pickle.dump(scaler, open(SCALER_NAME + ".pickle", "wb"))

X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
#######################################################################
# CREATE / LOAD / FIT MODEL
#######################################################################

MODEL_NAME = 'Canada-SK-model'

try:
    clf = pickle.load(open(MODEL_NAME + ".pickle", "rb"))
    from sklearn.utils.validation import check_is_fitted
    check_is_fitted(clf)
    print("Loaded existing model from", MODEL_NAME + ".pickle")

except (OSError, IOError):
    clf = None

except Exception:
    print("Pickle exists but model is not fitted — retraining.")
    clf = None

if clf is None:
    base_clf = MLPClassifier(
        solver             = 'adam',
        activation         = 'relu',
        alpha              = 0.0001,
        hidden_layer_sizes = (math.floor(LABEL_COL / 2),),
        learning_rate      = 'constant',
        max_iter           = 100,
        random_state       = 69420,
        warm_start         = True,
        verbose            = True,
        shuffle            = True,
        early_stopping     = True,
        validation_fraction= 0.1
    )
    parameter_space = {
        'hidden_layer_sizes': [(LABEL_COL,), (LABEL_COL*2,), (math.floor(LABEL_COL/2),)],
        'max_iter': [5, 10, 25, 50, 100, 150],
        'solver': ['sgd', 'adam', 'lbfgs'],
    }
    cv = GridSearchCV(base_clf, parameter_space, n_jobs=-1, cv=3)
    history = cv.fit(X_train, y_train)
    clf = cv.best_estimator_
    pickle.dump(clf, open(MODEL_NAME + ".pickle", "wb"))
    pickle.dump(history, open(MODEL_NAME + "_Loss.pickle", "wb"))
    print('Best parameters found:\n', cv.best_params_)

In [ ]:
#######################################################################
# SAMPLE PREDICTION DATA FROM GEE (PRED_YEAR) AND CLASSIFY
#######################################################################

print(f"Sampling {NUM_SAMPLES_PRED} prediction pixels (year={PRED_YEAR}) ...")

pred_image = (
    pred_embedding
    .addBands(pred_aafc.select('landcover').rename('aafc_label'))
)

pred_fc = pred_image.sample(
    region    = PRED_AOI,
    scale     = SAMPLE_SCALE,
    numPixels = NUM_SAMPLES_PRED,
    seed      = 0,
    geometries= True,
    dropNulls = True,
    tileScale = 8       # tiles the computation to stay under GEE memory limits
)

pred_df = geemap.ee_to_df(pred_fc, remove_geom=False)

pred_col_map = {b: i for i, b in enumerate(EMBEDDING_BANDS)}
pred_col_map['aafc_label'] = LABEL_COL
pred_df = pred_df.rename(columns=pred_col_map)

# Same remapping as training
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 132.0, 131.0, pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 138.0, 137.0, pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 139.0, 137.0, pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(
    ~pred_df[LABEL_COL].isin(SPECIFIC_CROPS), 500.0, pred_df[LABEL_COL]
)

desis_df_all = pred_df.copy()
labels_all   = pred_df[LABEL_COL].values

feature_cols = list(range(NUM_EMBEDDING_BANDS))
feat_scaled  = scaler.transform(imp.transform(pred_df[feature_cols]))
predictions  = clf.predict(feat_scaled)

print("\nPREDICTION DISTRIBUTION:")
print(pd.DataFrame(predictions).value_counts(normalize=True) * 100)
print("\nACTUAL (AAFC) DISTRIBUTION:")
print(pd.DataFrame(labels_all).value_counts(normalize=True) * 100)
print("\nAccuracy:", str(accuracy_score(predictions, labels_all) * 100) + "%")

In [ ]:
#######################################################################
# INITIAL PREDICTION ERROR MATRIX
#######################################################################

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(labels_all, predictions, labels=SPECIFIC_CROPS)
confusion_df = pd.DataFrame(cm, columns=SPECIFIC_CROPS, index=SPECIFIC_CROPS).astype(object)

confusion_df.loc['Total', :] = confusion_df.sum(axis=0)
confusion_df.loc[:, 'Total'] = confusion_df.sum(axis=1)

overall_acc = str(round(
    sum(int(confusion_df.loc[i, i]) for i in SPECIFIC_CROPS)
    / int(confusion_df.loc['Total', 'Total']) * 100, 2)) + "%"

confusion_df['User Acc'] = pd.Series(dtype=object, index=confusion_df.index)
for idx in SPECIFIC_CROPS:
    confusion_df.loc[idx, 'User Acc'] = str(round(
        int(confusion_df.loc[idx, idx]) / int(confusion_df.loc[idx, 'Total']) * 100, 2)) + "%"
confusion_df.loc['Total', 'User Acc'] = None

confusion_df.loc['Producer Acc', :] = None
for idx in SPECIFIC_CROPS:
    confusion_df.loc['Producer Acc', idx] = str(round(
        int(confusion_df.loc[idx, idx]) / int(confusion_df.loc['Total', idx]) * 100, 2)) + "%"
confusion_df.loc['Producer Acc', 'User Acc'] = overall_acc

confusion_df = confusion_df.rename(
    columns={c: CROP_LABELS.get(c, str(c)) for c in SPECIFIC_CROPS},
    index  ={c: CROP_LABELS.get(c, str(c)) for c in SPECIFIC_CROPS}
).rename(columns={'User Acc': 'User Accuracy'},
         index  ={'Producer Acc': 'Producer Accuracy'})

confusion_df.to_csv(MODEL_NAME + '_accuracy_scores.csv')
confusion_df

In [ ]:
#######################################################################
# GRAIN SUB-CLASSIFIER: Spring Wheat (131) vs Barley (137)
# These two look spectrally similar — a dedicated classifier improves separation
#######################################################################

SECONDARY_CROPS = [131.0, 137.0]

grain_df = AAFC_df.copy()
grain_df = grain_df.loc[grain_df[LABEL_COL].isin(SECONDARY_CROPS)]
grain_df = grain_df.reset_index(drop=True)

grain_train = scaler.transform(imp.transform(grain_df.loc[:, grain_df.columns != LABEL_COL]))
grain_label = grain_df[LABEL_COL]

grain_clf = MLPClassifier(
    solver             = 'adam',
    activation         = 'relu',
    alpha              = 0.0001,
    hidden_layer_sizes = (math.floor(LABEL_COL / 2),),
    learning_rate      = 'constant',
    max_iter           = 100,
    random_state       = 69420,
    warm_start         = True,
    verbose            = False,
    shuffle            = True,
    early_stopping     = True,
    validation_fraction= 0.1
)
grain_clf.fit(grain_train, grain_label)
print("Grain sub-classifier trained on", len(grain_df), "samples.")

In [ ]:
#######################################################################
# APPLY GRAIN SUB-CLASSIFIER TO PREDICTION SET
#######################################################################

desis_df_all[LABEL_COL] = predictions

new       = desis_df_all.copy()
le_grains = new.loc[new[LABEL_COL].isin(SECONDARY_CROPS)]

if len(le_grains) > 0:
    grain_features   = le_grains[list(range(NUM_EMBEDDING_BANDS))]
    grain_input      = scaler.transform(imp.transform(grain_features))
    grains_predict   = grain_clf.predict(grain_input)
    desis_df_all.loc[desis_df_all[LABEL_COL].isin(SECONDARY_CROPS), LABEL_COL] = grains_predict
    print(f"Re-classified {len(le_grains):,} wheat/barley pixels.")
else:
    print("No wheat/barley pixels found in predictions — sub-classifier not applied.")

In [ ]:
FINAL_PREDICTIONS = desis_df_all[LABEL_COL]
desis_df_all[LABEL_COL].value_counts()

In [ ]:
#######################################################################
# FINAL ACCURACY + WRITE CLASSIFIED POINTS TO CSV
#######################################################################

labels_all  = pred_df[LABEL_COL].values
predictions = FINAL_PREDICTIONS.values

pred_df['prediction'] = FINAL_PREDICTIONS.values
pred_df.to_csv(MODEL_NAME + '_CLASSIFIED.csv', index=False)
print(f"Saved: {MODEL_NAME}_CLASSIFIED.csv")

print("\nFINAL PREDICTION DISTRIBUTION:")
print(pd.DataFrame(predictions).value_counts(normalize=True) * 100)
print("\nACTUAL (AAFC) DISTRIBUTION:")
print(pd.DataFrame(labels_all).value_counts(normalize=True) * 100)
print("\nAccuracy:", str(accuracy_score(predictions, labels_all) * 100) + "%")

In [ ]:
#######################################################################
# FINAL ERROR MATRIX
#######################################################################

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(labels_all, predictions, labels=SPECIFIC_CROPS)
confusion_df = pd.DataFrame(cm, columns=SPECIFIC_CROPS, index=SPECIFIC_CROPS).astype(object)

confusion_df.loc['Total', :] = confusion_df.sum(axis=0)
confusion_df.loc[:, 'Total'] = confusion_df.sum(axis=1)

overall_acc = str(round(
    sum(int(confusion_df.loc[i, i]) for i in SPECIFIC_CROPS)
    / int(confusion_df.loc['Total', 'Total']) * 100, 2)) + "%"

confusion_df['User Acc'] = pd.Series(dtype=object, index=confusion_df.index)
for idx in SPECIFIC_CROPS:
    confusion_df.loc[idx, 'User Acc'] = str(round(
        int(confusion_df.loc[idx, idx]) / int(confusion_df.loc[idx, 'Total']) * 100, 2)) + "%"
confusion_df.loc['Total', 'User Acc'] = None

confusion_df.loc['Producer Acc', :] = None
for idx in SPECIFIC_CROPS:
    confusion_df.loc['Producer Acc', idx] = str(round(
        int(confusion_df.loc[idx, idx]) / int(confusion_df.loc['Total', idx]) * 100, 2)) + "%"
confusion_df.loc['Producer Acc', 'User Acc'] = overall_acc

confusion_df = confusion_df.rename(
    columns={c: CROP_LABELS.get(c, str(c)) for c in SPECIFIC_CROPS},
    index  ={c: CROP_LABELS.get(c, str(c)) for c in SPECIFIC_CROPS}
).rename(columns={'User Acc': 'User Accuracy'},
         index  ={'Producer Acc': 'Producer Accuracy'})

confusion_df.to_csv(MODEL_NAME + '_accuracy_scores_FINAL.csv')
confusion_df

In [ ]:
#######################################################################
# DOWNLOAD EMBEDDING FROM GEE AND CLASSIFY EVERY PIXEL
#######################################################################

OUT_DIR        = './Canada_SK/'
os.makedirs(OUT_DIR, exist_ok=True)

embedding_tif  = OUT_DIR + f'{PRED_YEAR}_embedding.tif'
classified_tif = OUT_DIR + f'{PRED_YEAR}_classified.tif'

# Download embedding (cached after first run)
if not os.path.exists(embedding_tif):
    print(f"Downloading {PRED_YEAR} embedding from GEE at {SAMPLE_SCALE} m resolution ...")
    geemap.ee_export_image(
        pred_embedding,
        filename      = embedding_tif,
        scale         = SAMPLE_SCALE,
        region        = PRED_AOI,
        file_per_band = False,
    )
    print("Download complete.")
else:
    print(f"Using cached embedding: {embedding_tif}")

dataset          = rasterio.open(embedding_tif)
dataset_meta     = dataset.meta
H, W             = dataset.shape
NUM_RASTER_BANDS = dataset.count
print(f"Raster: {W} x {H} = {W*H:,} pixels  |  {NUM_RASTER_BANDS} bands")

pixels = np.array([dataset.read(b) for b in range(1, NUM_RASTER_BANDS + 1)]) \
           .reshape(NUM_RASTER_BANDS, -1).T

feat_scaled = scaler.transform(imp.transform(pixels[:, :NUM_EMBEDDING_BANDS]))

print(f"Classifying {H*W:,} pixels ...")
raster_predictions = clf.predict(feat_scaled).astype(float)

# Apply grain sub-classifier to wheat/barley pixels
grain_mask = np.isin(raster_predictions, SECONDARY_CROPS)
if grain_mask.any():
    raster_predictions[grain_mask] = grain_clf.predict(feat_scaled[grain_mask]).astype(float)

print("\nClassification distribution:")
unique, counts = np.unique(raster_predictions, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {CROP_LABELS.get(u, str(u))}: {c:,} px  ({c/len(raster_predictions)*100:.1f}%)")

In [ ]:
#######################################################################
# WRITE CLASSIFIED IMAGE TO GEOTIFF + PREVIEW
#######################################################################

out_meta = dataset_meta.copy()
out_meta.update({'count': 1, 'dtype': 'float32', 'nodata': -1})

with rasterio.open(classified_tif, 'w', **out_meta) as dst:
    dst.write(raster_predictions.reshape(H, W).astype('float32'), 1)

print(f"Saved: {classified_tif}")
print(f"  Size : {W} x {H} pixels")
print(f"  CRS  : {dataset_meta.get('crs')}")

# Color-mapped preview
sorted_crops = sorted(SPECIFIC_CROPS)
val_idx      = {v: i for i, v in enumerate(sorted_crops)}
cmap         = plt.cm.get_cmap('tab10', len(sorted_crops))
indexed      = np.vectorize(val_idx.get)(raster_predictions).reshape(H, W)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(indexed, cmap=cmap, vmin=0, vmax=len(sorted_crops) - 1, interpolation='nearest')
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(val_idx[v])) for v in sorted_crops]
ax.legend(handles, [CROP_LABELS.get(v, str(v)) for v in sorted_crops],
          loc='lower right', fontsize=9)
ax.set_title(f'Predicted Crop Classification — Saskatchewan {PRED_YEAR}  (trained on {TRAIN_YEAR})')
ax.axis('off')
plt.tight_layout()
preview_path = classified_tif.replace('.tif', '_preview.png')
plt.savefig(preview_path, dpi=150)
plt.show()
print(f"Preview saved: {preview_path}")